# IA Hybride – Séance 2 : démonstration RAG hybride

ISIS Castres · 5e année DSIA

On construit pas à pas un assistant question-réponse sur des notices de médicaments, puis on l'hybride avec un petit graphe de connaissances et des règles de sécurité.

1. Corpus et jeu de questions
2. Recherche lexicale (BM25)
3. Recherche dense (embeddings)
4. Recherche hybride et évaluation (rappel@k)
5. Génération avec citations et garde-fou de refus
6. Hybridation : graphe d'interactions et règle de sécurité

> **Attention.** Le corpus est **fictif** : les médicaments, doses et interactions sont inventés pour l'exercice. Ne pas utiliser comme information médicale.

**Dépendances** : `numpy`, `scikit-learn`. Optionnel : `sentence-transformers` (embeddings de meilleure qualité, téléchargement d'un modèle au premier lancement) et [Ollama](https://ollama.com) avec un petit modèle local pour la génération. Le notebook fonctionne sans ces deux options.

In [1]:
import re, math, json
import numpy as np
from collections import Counter

np.set_printoptions(precision=3, suppress=True)

## 1. Corpus et jeu de questions

In [ ]:
corpus = [
    {"id": "zen-1", "doc": "Zentaprine", "texte": "La zentaprine est un antalgique indiqué dans les douleurs légères à modérées et la fièvre chez l'adulte."},
    {"id": "zen-2", "doc": "Zentaprine", "texte": "Posologie de la zentaprine chez l'adulte : 500 mg par prise, au maximum 3 g par jour, en respectant 4 heures entre deux prises."},
    {"id": "zen-3", "doc": "Zentaprine", "texte": "La zentaprine est contre-indiquée en cas d'insuffisance hépatique sévère. Une surveillance du foie est recommandée en cas de consommation d'alcool."},
    {"id": "lor-1", "doc": "Lorvaxol", "texte": "Le lorvaxol est un anticoagulant oral utilisé pour prévenir la formation de caillots après une chirurgie orthopédique."},
    {"id": "lor-2", "doc": "Lorvaxol", "texte": "Le lorvaxol augmente le risque de saignement. L'association avec le brufénide, un anti-inflammatoire, est déconseillée."},
    {"id": "lor-3", "doc": "Lorvaxol", "texte": "En cas d'oubli d'une dose de lorvaxol, ne pas doubler la dose suivante. Contacter le médecin en cas de saignement inhabituel."},
    {"id": "bru-1", "doc": "Brufénide", "texte": "Le brufénide est un anti-inflammatoire non stéroïdien utilisé contre les douleurs articulaires et l'inflammation."},
    {"id": "bru-2", "doc": "Brufénide", "texte": "Le brufénide doit être pris au cours d'un repas. Il peut provoquer des brûlures d'estomac et des ulcères."},
    {"id": "bru-3", "doc": "Brufénide", "texte": "Le brufénide est contre-indiqué à partir du sixième mois de grossesse et en cas d'insuffisance rénale sévère."},
    {"id": "met-1", "doc": "Métarzine", "texte": "La métarzine est un antidiabétique oral qui diminue le taux de sucre dans le sang chez les patients atteints de diabète de type 2."},
    {"id": "met-2", "doc": "Métarzine", "texte": "La métarzine ne doit pas être utilisée si la fonction rénale est très diminuée. Une prise de sang régulière contrôle les reins."},
    {"id": "met-3", "doc": "Métarzine", "texte": "Effets indésirables fréquents de la métarzine : nausées, diarrhée, goût métallique, surtout en début de traitement."},
]
# Questions de test : passage attendu (None = aucune réponse dans le corpus)
questions = [
    {"q": "Quelle est la dose maximale de zentaprine par jour ?", "attendu": "zen-2"},
    {"q": "Peut-on prendre du brufénide avec le lorvaxol ?", "attendu": "lor-2"},
    {"q": "Le brufénide est-il autorisé pendant la grossesse ?", "attendu": "bru-3"},
    {"q": "Quels sont les effets secondaires de la métarzine ?", "attendu": "met-3"},
    {"q": "Que faire si j'ai oublié de prendre mon anticoagulant ?", "attendu": "lor-3"},
    {"q": "Quel médicament fait baisser la glycémie ?", "attendu": "met-1"},
    {"q": "Le foie est-il un problème avec la zentaprine ?", "attendu": "zen-3"},
    {"q": "Quelle est la posologie du brufénide chez l'enfant ?", "attendu": None},
    {"q": "Le lorvaxol se prend-il avec du pamplemousse ?", "attendu": None},
]
textes = [p["texte"] for p in corpus]
ids = [p["id"] for p in corpus]
print(len(corpus), "passages,", len(questions), "questions")

## 2. Recherche lexicale : BM25

BM25 note un passage $d$ pour une requête $q$ par

$$\text{BM25}(q, d) = \sum_{t \in q} \text{idf}(t) \cdot \frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1\left(1 - b + b\,\frac{|d|}{\overline{|d|}}\right)}$$

avec $f(t,d)$ le nombre d'occurrences du terme $t$ dans $d$, $\text{idf}(t) = \log\left(1 + \frac{N - n_t + 0{,}5}{n_t + 0{,}5}\right)$, $k_1 = 1{,}5$ et $b = 0{,}75$.

In [ ]:
STOP = set("le la les l de des du d un une et en à au aux est sont pour par avec ne pas se si on il elle qui que quel quelle quels quelles ce cas peut chez dans sur être doit j mon".split())

def tokeniser(texte):
    mots = re.findall(r"[a-zàâçéèêëîïôûùüÿœ0-9]+", texte.lower())
    return [m for m in mots if m not in STOP]

docs_tok = [tokeniser(t) for t in textes]
print(docs_tok[1])

In [ ]:
class BM25:
    def __init__(self, docs_tok, k1=1.5, b=0.75):
        self.docs, self.k1, self.b = docs_tok, k1, b
        self.N = len(docs_tok)
        self.longueur_moy = np.mean([len(d) for d in docs_tok])
        self.df = Counter(t for d in docs_tok for t in set(d))

    def idf(self, t):
        n = self.df.get(t, 0)
        return math.log(1 + (self.N - n + 0.5) / (n + 0.5))

    def scores(self, requete):
        q = tokeniser(requete)
        out = np.zeros(self.N)
        for i, d in enumerate(self.docs):
            tf = Counter(d)
            norm = self.k1 * (1 - self.b + self.b * len(d) / self.longueur_moy)
            out[i] = sum(self.idf(t) * tf[t] * (self.k1 + 1) / (tf[t] + norm) for t in q if t in tf)
        return out

bm25 = BM25(docs_tok)
s = bm25.scores("dose maximale zentaprine")
print(ids[int(np.argmax(s))])   # attendu : zen-2

## 3. Recherche dense

On encode passages et questions en vecteurs, puis on compare par similarité cosinus (séance 1).

- Si `sentence-transformers` est installé, on utilise un modèle multilingue pré-entraîné.
- Sinon, solution de repli : TF-IDF sur des n-grammes de caractères, réduit par SVD (analyse sémantique latente). C'est moins bon, mais cela tourne partout.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    modele = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    def encoder(liste):
        return modele.encode(liste, normalize_embeddings=True)
    MODE_DENSE = "sentence-transformers"
except Exception as e:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    vect = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
    X = vect.fit_transform(textes)
    svd = TruncatedSVD(n_components=min(10, len(textes) - 1), random_state=0).fit(X)
    def encoder(liste):
        Z = svd.transform(vect.transform(liste))
        return Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)
    MODE_DENSE = "repli TF-IDF + SVD"

E_passages = encoder(textes)
print("Mode dense :", MODE_DENSE, "| dimension :", E_passages.shape[1])

In [ ]:
def scores_denses(requete):
    q = encoder([requete])[0]
    return E_passages @ q

print(ids[int(np.argmax(scores_denses("Quel médicament fait baisser la glycémie ?")))])

## 4. Recherche hybride et évaluation

On combine les deux scores après normalisation min-max : $s = \alpha \, s_{\text{dense}} + (1-\alpha)\, s_{\text{BM25}}$.

On mesure le **rappel@k** : la part des questions (avec réponse) dont le passage attendu figure parmi les $k$ premiers.

In [ ]:
def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-12)

def scores_hybrides(requete, alpha=0.5):
    return alpha * minmax(scores_denses(requete)) + (1 - alpha) * minmax(bm25.scores(requete))

def rechercher(requete, k=3, methode="hybride", alpha=0.5):
    s = {"bm25": bm25.scores, "dense": scores_denses,
         "hybride": lambda r: scores_hybrides(r, alpha)}[methode](requete)
    ordre = np.argsort(-s)[:k]
    return [(ids[i], float(s[i])) for i in ordre]

rechercher("Peut-on prendre du brufénide avec le lorvaxol ?")

In [ ]:
def rappel_a_k(methode, k, alpha=0.5):
    qs = [q for q in questions if q["attendu"]]
    trouves = [q["attendu"] in [i for i, _ in rechercher(q["q"], k, methode, alpha)] for q in qs]
    return float(np.mean(trouves))

for methode in ["bm25", "dense", "hybride"]:
    print(methode, [round(rappel_a_k(methode, k), 2) for k in (1, 3)])

**Questions.**
1. Quelle question BM25 rate-t-il, et pourquoi ? (indice : « anticoagulant », « glycémie »)
2. Faites varier `alpha` de 0 à 1. Quelle valeur choisiriez-vous ? Sur combien de questions vous appuyez-vous pour décider ?
3. Si vous êtes en mode « repli », installez `sentence-transformers` (`pip install sentence-transformers`) et comparez : la recherche dense comprend-elle mieux « glycémie » ≈ « sucre dans le sang » ?

## 5. Génération avec citations et garde-fou

On construit une consigne qui impose de répondre uniquement d'après les passages et de citer `[n]`. Si Ollama tourne localement, on l'appelle ; sinon on produit une réponse **extractive** (la phrase la plus pertinente, citée), ce qui suffit pour la démonstration.

Garde-fou : si le meilleur score dense est sous un seuil, on refuse de répondre.

In [ ]:
def construire_consigne(question, passages):
    contexte = "\n".join(f"[{n+1}] ({pid}) {corpus[ids.index(pid)]['texte']}" for n, (pid, _) in enumerate(passages))
    return ("Tu es un assistant d'information sur les médicaments. Réponds uniquement à partir des passages "
            "ci-dessous et cite-les sous la forme [n]. Si les passages ne permettent pas de répondre, "
            "réponds exactement : « Je ne sais pas d'après les documents disponibles. »\n\n"
            f"Passages :\n{contexte}\n\nQuestion : {question}\nRéponse :")

def appeler_ollama(consigne, modele="llama3.2:3b"):
    import urllib.request
    req = urllib.request.Request("http://localhost:11434/api/generate",
                                 data=json.dumps({"model": modele, "prompt": consigne, "stream": False}).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read())["response"].strip()

def reponse_extractive(question, passages):
    pid = passages[0][0]
    return f"{corpus[ids.index(pid)]['texte']} [1]"

In [ ]:
SEUIL = 0.35   # à régler

def repondre(question, k=3):
    passages = rechercher(question, k, "hybride")
    meilleur_dense = float(np.max(scores_denses(question)))
    if meilleur_dense < SEUIL:
        return f"Je ne sais pas d'après les documents disponibles. (similarité max {meilleur_dense:.2f})"
    try:
        return appeler_ollama(construire_consigne(question, passages))
    except Exception:
        return reponse_extractive(question, passages)

for q in questions:
    print("Q :", q["q"]); print("R :", repondre(q["q"])); print()

**Questions.**
1. Les deux questions sans réponse (posologie enfant du brufénide, pamplemousse) sont-elles refusées ? Sinon, que renvoie le système et pourquoi est-ce dangereux ?
2. Un seuil sur la similarité suffit-il ? Proposez un autre garde-fou.

## 6. Hybridation : graphe d'interactions et règle de sécurité

Le RAG dépend de la formulation de la question. Pour les interactions médicamenteuses, on ajoute une connaissance **structurée** et une **règle de sécurité** appliquée en sortie, quelle que soit la réponse générée.

In [ ]:
# Mini-graphe de connaissances (fictif) : triplets (tête, relation, queue)
kg = [
    ("lorvaxol", "classe", "anticoagulant"),
    ("brufenide", "classe", "ains"),
    ("zentaprine", "classe", "antalgique"),
    ("metarzine", "classe", "antidiabetique"),
    ("anticoagulant", "interaction_deconseillee", "ains"),
]
SYNONYMES = {"brufénide": "brufenide", "métarzine": "metarzine"}

def medicaments_cites(texte):
    mots = [SYNONYMES.get(m, m) for m in tokeniser(texte)]
    connus = {h for h, r, _ in kg if r == "classe"}
    return sorted(set(mots) & connus)

def classe(med):
    return next(t for h, r, t in kg if h == med and r == "classe")

In [ ]:
def alertes_interactions(medicaments):
    alertes = []
    for i, a in enumerate(medicaments):
        for b in medicaments[i + 1:]:
            ca, cb = classe(a), classe(b)
            if (ca, "interaction_deconseillee", cb) in kg or (cb, "interaction_deconseillee", ca) in kg:
                alertes.append(f"ALERTE : {a} ({ca}) + {b} ({cb}) — association déconseillée")
    return alertes

alertes_interactions(["lorvaxol", "brufenide", "zentaprine"])

In [ ]:
def assistant_hybride(question):
    reponse = repondre(question)
    alertes = alertes_interactions(medicaments_cites(question))
    return reponse + ("\n" + "\n".join(alertes) if alertes else "")

print(assistant_hybride("Je prends du lorvaxol, puis-je ajouter du brufénide pour mon genou ?"))
print()
print(assistant_hybride("Lorvaxol et brufénide ensemble, c'est bon ?"))

**Questions.**
1. Où entre la connaissance dans cette architecture (séance 2, quatre points d'entrée) ? Où la placer dans la taxonomie de Kautz ?
2. Ajoutez au graphe un nouveau médicament de classe AINS. L'alerte fonctionne-t-elle sans rien changer au code ? Quel avantage par rapport au RAG seul ?
3. Quelles limites voyez-vous à la détection des médicaments par mots-clés ? Comment un LLM ou un dictionnaire de synonymes pourrait-il aider ?

---
Ce notebook est le point de départ des projets 2 (RAG sur notices) et 5 (GraphRAG avec validation humaine).